# Visiona3D: one YOLO26 model for objects + environment

Trains and evaluates the unified model (YOLO26n-seg trunk + instance head + new 8-class semantic head) **on this computer**.
Read `Future Work.md` first: it explains the goal, the data, the first-run results and the checks the model must pass before it replaces anything in Visiona.

All logic lives in `visiona_mtl/` (see `workflow.py`). Nothing downloads or trains until you switch it on below.

In [ ]:
import sys
from pathlib import Path

LIB = Path.cwd() if (Path.cwd() / "visiona_mtl").exists() else Path.cwd() / "notebooks"
sys.path.insert(0, str(LIB))
from visiona_mtl import workflow as W

RUN_DOWNLOADS = False   # True: fetch missing datasets (COCO, COCO-Stuff, ADE20K, sidewalk). Large; skips what is already on disk.
RUN_TRAINING = False    # True: train, evaluate and export (hours on a CPU). Resumes from the last checkpoint.
RUN_E2 = False          # True: also run E2 (first 6 layers frozen). Only after reading the E1 results.

cfg = W.make_config()   # imgsz=320, batch=4, 20 epochs x 500 steps; override any key, e.g. W.make_config(epochs=60, time_budget_hours=30)

## 1. Data

In [ ]:
if RUN_DOWNLOADS:
    W.fetch_coco_and_ade(cfg)
    W.fetch_sidewalk(cfg)       # gated dataset: needs your Hugging Face login and accepted terms
records = W.load_records(cfg)   # converts anything not yet converted to the 8 class labels

## 2. Model and baseline (the baseline is saved before any training)

In [ ]:
W.show_model(cfg)
if RUN_TRAINING:
    print(W.save_baseline(cfg))

## 3. Train E1 (backbone frozen). Checkpoints are written every epoch.

In [ ]:
if RUN_TRAINING:
    W.train(cfg, records)

## 4. Evaluate against the baseline, measure speed, write the final files

Read `work/checkpoints/<name>/evaluation.json` before doing anything else. Model-only timings are **not** the application's FPS.

In [ ]:
if RUN_TRAINING:
    W.evaluate(cfg, records)
    W.speed(cfg)
    W.write_final(cfg)

## 5. Optional: E2 (only after reading the E1 results)

In [ ]:
if RUN_TRAINING and RUN_E2:
    W.train(cfg, records, name="E2_partly_unfrozen", freeze=6)
    W.evaluate(cfg, records, name="E2_partly_unfrozen")